# 9. End-to-End Image Processing: M 13

In this User Guide Cookbook, you will learn how to process a single real dataset (M 13) through all the available pipelines in a single workflow.

Earlier notebooks introduced each pipeline stage on its own, with the reasoning behind it. Here, you will run every stage back to back against the same telescope images -- from raw calibration through a finished star catalog and extracted spectra -- so you can see how one stage's output becomes the next stage's input. You can run this notebook on its own; it loads its own copy of the sample data instead of requiring earlier notebooks to have run first.

You will see how to:
1. Initialize the `Astrometrics` high-level interface and load the bundled M 13 sample data.
2. Stack and calibrate raw images.
3. Perform astrometry and plate-solving.
4. Run differential photometry.
5. Extract 1D spectra.


In [1]:
from astrometricslib import Astrometrics, FilterType
from documentation.notebooks.astrometrics.scripts.sample_data_staging import (
    drop_stale_sample_data_frames,
    stage_m13_sample_data,
)
from wayfindinglib import Wayfinder

astrometrics = Astrometrics()

# A single Target entity holds both imaging (Luminance) and spectroscopy (SPEC)
# frames. Both are light frames of the same celestial object (M 13); the
# distinction between them is made downstream when stacking via kind.
# create() returns the existing target when there is one.
target = astrometrics.targets.create("M 13")
drop_stale_sample_data_frames(target)

sample_data_dir = stage_m13_sample_data()
new_paths = [
    str(path)
    for pattern in ("M_13_Light_Luminance_*.fits", "M_13_Light_Spectroscopy_*.fits")
    for path in sorted(sample_data_dir.glob(pattern))
]
# Reading each header, adding the frames and saving the target is one step.
# A file that is already listed is updated, not added twice.
astrometrics.targets.reindex_frames(target, paths=new_paths, role="LIGHT")

print(f"Loaded '{target.id}' with {len(target.frames)} registered frames.")

Loaded 'M 13' with 10 registered frames.


## 2. Inspecting Frame Records

The target contains both Luminance and slitless spectroscopy exposures.

- **Luminance frames** are taken through a clear filter passing visible light, feeding astrometry and photometry.
- **Spectroscopy frames** are slitless grating exposures spreading starlight into 1D spectra, carrying the `SPEC` filter.


In [2]:
luminance_frames = [f for f in target.frames if f.filter == FilterType.L]
spectroscopy_frames = [f for f in target.frames if f.filter == FilterType.SPEC]

print(f"Luminance frames: {len(luminance_frames)}")
print(f"Spectroscopy frames: {len(spectroscopy_frames)}")

Luminance frames: 5
Spectroscopy frames: 5


## 3. Stacking Luminance Frames (`stacking.stacked_image`)

We stack the Luminance exposures with `astrometrics.processing.stack`.

Stacking averages the frames together, which raises the signal relative to the noise. Before averaging, the pipeline applies Chauvenet rejection: for each pixel position, it discards any value sitting further from the average than a sample of that size should plausibly produce. This is what removes satellite trails and cosmic ray hits, which appear in one frame but not the rest.


In [3]:
print(f"Stacking Luminance frames for {target.id}...")
# stack() records the master frame on the target and saves it.
stacked_image = astrometrics.processing.stack(target, filter_name=FilterType.L).stacked_path
print(f"Luminance master stack: {stacked_image}")

Stacking Luminance frames for M 13...
Luminance master stack: /home/user/astrometrics/libraryIndex/frames/lights/M 13/M_13_L_Stacked.fits


## 4. Stacking Spectroscopy Frames (`spectral_stacking.stacked_image`)

We execute frame stacking on the slitless spectroscopy (`SPEC`) exposures using `astrometrics.processing.stack`.

In [4]:
print(f"Stacking Spectroscopy frames for {target.id}...")
stacked_spectral_image = astrometrics.processing.stack(target, kind="spectral").stacked_path
print(f"Spectroscopy master stack: {stacked_spectral_image}")

Stacking Spectroscopy frames for M 13...


Spectroscopy master stack: /home/user/astrometrics/libraryIndex/frames/lights/M 13/M_13_SPEC_Stacked.fits


## 5. Running Astrometry Analysis

We run plate solving and catalog cross-matching on the master Luminance image with `astrometrics.processing.process_target` with the astrometry stage.

Plate solving works out which patch of sky an image shows, by matching the pattern of stars in it against a catalog. The answer is written into the image's FITS header as a WCS (World Coordinate System), the mapping that turns a pixel position into a sky coordinate.

This step needs one of two things: a local `solve-field` install whose index files match how much sky your camera and telescope see, or an astrometry.net API key. See `documentation/Installation.md`.


In [5]:
print(f"Running astrometry plate-solving for {target.id}...")
astrometry_result = astrometrics.processing.process_target(target, stages=["astrometry"]).results[
    "astrometry"
]
print(f"Astrometry solved: {astrometry_result.get('wcs') is not None}")
print(f"Stars detected: {len(astrometry_result.get('stellar_objects', []))}")

Running astrometry plate-solving for M 13...


The current archive release 3.15 contains two breaking changes. Please see https://www.cosmos.esa.int/web/gaia/news#GACSrelease315 for details. In case of questions or feedback, please consult the Gaia helpdesk at https://www.cosmos.esa.int/web/gaia/gaia-helpdesk.


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


INFO: Query finished. [astroquery.utils.tap.core]


Astrometry solved: True
Stars detected: 500


## 6. Running Photometry Analysis

We perform aperture photometry and ensemble calibration across the Luminance frame sequence using the photometry stage of `astrometrics.processing.process_target`.

In [6]:
print(f"Running ensemble photometry for {target.id}...")
photometry_result = astrometrics.processing.process_target(
    target,
    stages=["photometry"],
    photometry={"filter_name": FilterType.L, "frames": luminance_frames, "use_astrometry_seed": True},
).results["photometry"]
print(f"Photometry status: {photometry_result.get('status')}")
print(f"Stars found: {photometry_result.get('starsFound', 0)}")

Running ensemble photometry for M 13...


Photometry status: completed
Stars found: 482


## 7. Running Spectroscopy Analysis

We detect zero-order star centroids and extract calibrated 1D spectra from the dispersed spectral image using the spectroscopy stage of `astrometrics.processing.process_target`.

In [7]:
print(f"Running spectroscopy analysis for {target.id}...")
spectroscopy_result = astrometrics.processing.process_target(target, stages=["spectroscopy"]).results[
    "spectroscopy"
]
spectral_stars = spectroscopy_result.get("stellar_objects", [])
print(f"Spectroscopy extracted {len(spectral_stars)} spectra.")

summary = target.quality.spectroscopy
failed_gates = [gate.name for gate in summary.gates if gate.status.value == "failed"] if summary else []
status = "failed gates: " + ", ".join(failed_gates) if failed_gates else "completed"
print(f"Spectroscopy Pipeline Status: {status}")

# Each star keeps its match score and its gap to another class.
print("\nSelf-determined spectral types (lower match score is closer):")
for star in spectral_stars:
    spectrum = astrometrics.stars.get_spectrum(star.id)
    if spectrum is not None and spectrum.self_determined_spectral_type_rms is not None:
        print(
            f"  {star.id}: {spectrum.self_determined_spectral_type} "
            f"(match score={spectrum.self_determined_spectral_type_rms:.3f}, "
            f"gap to next class={spectrum.rms_gap_to_next_class})"
        )

# What to do about it is the Wayfinder's call: it reads the measurements and
# answers with statements (warnings, recommendations), each naming its rule.
advice = Wayfinder().review.get_advisory(kind="quality", target=target)
print(f"\nThe Wayfinder's advice for {target.id}: {len(advice.statements)} statement(s)")
for statement in advice.statements[:5]:
    print(f"  [{statement.kind.value}] {statement.rule}: {statement.plain_message or statement.message}")

Running spectroscopy analysis for M 13...


Spectroscopy extracted 431 spectra.
Spectroscopy Pipeline Status: failed gates: spectral_classification, feature_significance, processing_quality, wavelength_scale, gaia_xp_agreement

Self-determined spectral types (lower match score is closer):
  HD 150998: K5V (match score=0.135, gap to next class=0.042919126334775376)
  HD 150679: A7V (match score=0.052, gap to next class=0.030605765918301074)
  Gaia DR3 1327617929278818176: K7V (match score=0.134, gap to next class=0.018669913477637468)
  HD 150293: K2V (match score=0.072, gap to next class=0.043510970324478024)
  HD 151023: K2V (match score=0.050, gap to next class=0.05161437743782477)
  BD+36  2775: K5V (match score=0.127, gap to next class=0.0777740026511019)
  HD 151086: K2V (match score=0.104, gap to next class=0.04399402898731271)
  BD+35  2851: K7V (match score=0.127, gap to next class=0.012929084511670519)
  BD+36  2764: K0V (match score=0.102, gap to next class=0.007017606552449637)
  TYC 2588-1167-1: K4V (match score=0.12

  Gaia DR3 1328056874936120192: M2V (match score=0.281, gap to next class=0.0079583725976769)
  NGC  6205   775: M3V (match score=0.349, gap to next class=0.14044015784987796)
  2MASS J16414398+3622338: M5V (match score=0.390, gap to next class=0.4213246493767339)
  NGC  6205   848: M2V (match score=0.324, gap to next class=0.10437100655354792)
  NGC  6205   991: M4V (match score=0.431, gap to next class=0.30791601209402447)


  FIELD_J250.4398+36.3886: M4V (match score=0.417, gap to next class=0.33768302937421873)
  NGC  6205   638: M2V (match score=0.265, gap to next class=0.015426494613049457)


  NGC  6205   955: M3V (match score=0.392, gap to next class=0.11186346413419218)
  FIELD_J250.4323+36.4117: M3V (match score=0.327, gap to next class=0.14529266700711868)
  NGC  6205   833: M4V (match score=0.414, gap to next class=0.330810407859089)



The Wayfinder's advice for M 13: 15 statement(s)
  [warning] overexposed_setting: In the 30 s frames, the brightest star is roughly 1.9 times brighter than the camera can record, so it is likely washed out. Use 12.4 s or less.
  [recommendation] calibration_gap: No darks in your library match these frames, so the calibration may be off. Take new ones at gain 0, offset 0, bin 1, 30 s, -10 C.
  [recommendation] calibration_gap: No flats in your library match these frames, so the calibration may be off. Take new ones at gain 0, offset 0, bin 1, filter Luminance.
  [insufficient_data] flat_noise_limits_stack: We could not check 'flat noise limits stack' yet because some measurements are missing. The reason says what to measure.
  [insufficient_data] master_flat_noisy: We could not check 'master flat noisy' yet because some measurements are missing. The reason says what to measure.


> **Shortcut:** Steps 5-7 above ran astrometry, photometry, and spectroscopy
> one call at a time so each step's own options (like the Luminance-only
> `frames=luminance_frames` filter in step 6) are visible individually. When
> no per-stage customization is needed, `astrometrics.processing.process_target`
> runs the same three stages, in the same order, in one call.

In [8]:
# The shorter equivalent of steps 5-7, with no per-stage overrides:
# results = astrometrics.processing.process_target(target)
# results["astrometry"], results["photometry"], results["spectroscopy"]

## 8. Querying Enriched Stellar Catalog

We list and inspect the solved stellar objects associated with **M 13** via `astrometrics.stars.query`.

In [9]:
stellar_objects = astrometrics.stars.query(target_id=target.id, detail="objects", limit=None).objects
print(f"Retrieved {len(stellar_objects)} stellar objects for target {target.id}.")
if stellar_objects:
    sample_star = stellar_objects[0]
    print(f"Sample Star: ID={sample_star.id}, Name={sample_star.name}, Mag={sample_star.magnitude}")

Retrieved 562 stellar objects for target M 13.
Sample Star: ID=1SWASP J164159.14+363818.5, Name=1SWASP J164159.14+363818.5, Mag=11.880000114440918


## 9. Visualizing Results with Interactive Dashboards

We visualize the comprehensive multi-modal analysis results using the high-level `astrometrics.visualization` tools.

### 9.1 Interactive Target Dashboard

The target dashboard displays the FITS image with astrometric star overlays alongside side panels for photometry and spectroscopy that update dynamically when clicking stars.

In [10]:
print(f"Rendering interactive target dashboard for {target.id}...")
figure = astrometrics.visualization.plot("dashboard", target)
figure.show()

Rendering interactive target dashboard for M 13...


/tmp/ipykernel_1377249/3510017542.py:3: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  figure.show()


### 9.2 Star Multi-Modal Analysis Dashboard

The star dashboard displays a two-panel visualization for a single star showing its photometric light curve and calibrated 1D spectrum.

In [11]:
if stellar_objects:
    selected_star = stellar_objects[0]
    print(f"Rendering star dashboard for {selected_star.id}...")
    star_figure = astrometrics.visualization.plot("star", star=selected_star)
    star_figure.show()
else:
    print("No stellar objects were identified for this target; skipping the star dashboard.")

Rendering star dashboard for 1SWASP J164159.14+363818.5...


/tmp/ipykernel_1377249/7039746.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  star_figure.show()


## Summary

You have learned how to:
1. Load a real dataset containing multiple filter types under a unified target.
2. Execute Chauvenet rejection stacking for luminance and spectroscopy frames via `astrometrics.processing.stack`.
3. Solve astrometry, extract photometry, and calibrate 1D spectra with the astrometry, photometry and spectroscopy stages of `astrometrics.processing.process_target`.
4. Inspect multi-modal results using `astrometrics.visualization.plot` with `kind="dashboard"` and `kind="star"`.
